# 📘 Modèle Pyomo généré automatiquement

## 📦 Imports

In [ ]:
from pyomo.environ import *
from pyomo.opt import SolverFactory
import pandas as pd

## 💾 Chargement des données
Les données `.dat` sont chargées nativement par Pyomo via `model.create_instance(...)` dans la section du modèle.

## 🔹 Model

In [ ]:
from pyomo.environ import *

model = AbstractModel()

## 🔹 Sets

In [ ]:
model.ALIMENTS = Set()
model.INGREDIENTS = Set()
model.ARCS = Set(dimen=2, initialize=lambda m: [(i0,i1) for i0 in m.ALIMENTS for i1 in m.INGREDIENTS])

## 🔹 Parameters

In [ ]:
model.Prix = Param(model.ALIMENTS, within=NonNegativeReals)
model.Calories = Param(model.ALIMENTS, within=NonNegativeReals)
model.DIETEJOUR = Param(model.INGREDIENTS, within=NonNegativeReals)
model.QTEING = Param(model.ALIMENTS, model.INGREDIENTS, within=NonNegativeReals)

## 🔹 Variables

In [ ]:
model.X = Var(model.ALIMENTS, domain=NonNegativeReals)

## 🔹 Data

In [ ]:
model = model.create_instance('../data/Regime_data.dat')

## 🔹 Constraints

In [ ]:
model.c0 = Constraint(expr=sum(model.Calories[a]*model.X[a] for a in model.ALIMENTS) >= 500)
model.c_for_0 = ConstraintList()
for i in model.INGREDIENTS:
    model.c_for_0.add(sum(model.QTEING[a, i] * model.X[a] for a in model.ALIMENTS) >= model.DIETEJOUR[i])

## 🔹 Objective

In [ ]:
model.obj = Objective(expr=sum(model.Prix[a] * model.X[a] for a in model.ALIMENTS), sense=minimize)

## ⚙️ Résolution du modèle

In [ ]:
solver = SolverFactory('highs')
result = solver.solve(model, tee=True)

print('✅ Solver status:', result.solver.status)
print('✅ Termination condition:', result.solver.termination_condition)

## 🎯 Valeur de la fonction objective

In [ ]:
for obj in model.component_objects(Objective, active=True):
    print(f'Objectif: {obj.name}')
    print(f'Valeur optimale: {obj():.4f}')
    print(f'Sens: {"Minimisation" if obj.sense == minimize else "Maximisation"}')

## 📊 Valeurs optimales des variables

In [ ]:
# Extraction des résultats dans un DataFrame
results_data = []
for v in model.component_objects(Var, active=True):
    for index in v:
        results_data.append({
            'Variable': v.name,
            'Index': str(index) if index != None else '-',
            'Valeur': v[index].value
        })

df_results = pd.DataFrame(results_data)
# Filtrer les valeurs non-nulles pour plus de clarté
df_results = df_results[df_results['Valeur'].notna()]
df_results = df_results[df_results['Valeur'] != 0]
df_results.style.format({'Valeur': '{:.4f}'}).set_caption('Variables de décision optimales')